Import Libraries

In [14]:
import pandas as pd
from pathlib import Path
import statsmodels.api as sm

Set-up

In [15]:
# Get root directory and data directory
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_PATH = (PROJECT_ROOT / "data" / "processed" / "retailhero_causal_forest_features.csv")

RANDOM_STATE = 1213

In [16]:
# Load data
print("=" * 60)
print("Linear Probability Models")
print("=" * 60)

print(f"\nLoading data from:")
print(PROCESSED_DATA_PATH.resolve())

df = pd.read_csv(PROCESSED_DATA_PATH)

print(f"\nFull dataset shape: {df.shape}")

Linear Probability Models

Loading data from:
/Users/edenaong/Downloads/Y4S1/DSE4101/DSE4101-CausalForest-Grp2/data/processed/retailhero_causal_forest_features.csv

Full dataset shape: (200039, 25)


In [17]:
# Define Y, D
Y = df["target"].astype(int)
D = df["treatment"].astype(int)

# Linear Probability Models

## Model 1: Prior Total Spending

$$
Y_i = \beta_0 + \beta_D D_i + \beta_1 \text{Spend}_i^{\mathrm{std}} + \gamma_1 (D_i \times \text{Spend}_i^{\mathrm{std}}) + \varepsilon_i
$$

In [18]:
# Standardise total spending
spend_mean = df["total_spend"].mean()
spend_std = df["total_spend"].std()

spend_z = (df["total_spend"] - spend_mean) / spend_std

# Construct regression dataset
model1_df = pd.DataFrame({
    "target": Y,
    "treatment": D,
    "total_spend_std": spend_z
})

model1_df["treatment_x_total_spend"] = model1_df["treatment"] * model1_df["total_spend_std"]

# Define regressors
X_lpm1 = model1_df[["treatment", "total_spend_std", "treatment_x_total_spend"]]
X_lpm1 = sm.add_constant(X_lpm1)

# Fit LPM with heteroskedasticity-robust standard errors
lpm_model1 = sm.OLS(model1_df["target"], X_lpm1).fit(cov_type="HC3")

print("\n" + "=" * 40)
print("Pre-specified LPM: Prior Total Spending")
print("=" * 40)
print(lpm_model1.summary())


Pre-specified LPM: Prior Total Spending
                            OLS Regression Results                            
Dep. Variable:                 target   R-squared:                       0.062
Model:                            OLS   Adj. R-squared:                  0.062
Method:                 Least Squares   F-statistic:                     667.0
Date:                Sat, 24 Oct 2026   Prob (F-statistic):               0.00
Time:                        18:53:42   Log-Likelihood:            -1.3284e+05
No. Observations:              200039   AIC:                         2.657e+05
Df Residuals:                  200035   BIC:                         2.657e+05
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------

p-value = 0.031 < 0.05

There is statistically significant evidence that treatment effect heterogenity is associated with prior total spending.

## Model 2: Number of Transactions

$$
Y_i = \beta_0 + \beta_D D_i + \beta_1 \text{Transactions}_i^{\mathrm{std}} + \gamma_1 (D_i \times \text{Transactions}_i^{\mathrm{std}}) + \varepsilon_i
$$

In [19]:
# Standardise number of transactions
transactions_mean = df["num_transactions"].mean()
transactions_std = df["num_transactions"].std()

transactions_z = (
    df["num_transactions"] - transactions_mean
) / transactions_std

# Construct regression dataset
model2_df = pd.DataFrame({
    "target": Y,
    "treatment": D,
    "num_transactions_std": transactions_z
})

model2_df["treatment_x_num_transactions"] = model2_df["treatment"] * model2_df["num_transactions_std"]

# Define regressors
X_lpm2 = model2_df[["treatment", "num_transactions_std", "treatment_x_num_transactions"]]
X_lpm2 = sm.add_constant(X_lpm2)

# Fit LPM with robust standard errors
lpm_model2 = sm.OLS(model2_df["target"], X_lpm2).fit(cov_type="HC3")

print("\n" + "=" * 50)
print("Pre-specified LPM: Prior Number of Transactions")
print("=" * 50)
print(lpm_model2.summary())


Pre-specified LPM: Prior Number of Transactions
                            OLS Regression Results                            
Dep. Variable:                 target   R-squared:                       0.128
Model:                            OLS   Adj. R-squared:                  0.128
Method:                 Least Squares   F-statistic:                     6924.
Date:                Sat, 24 Oct 2026   Prob (F-statistic):               0.00
Time:                        18:53:42   Log-Likelihood:            -1.2552e+05
No. Observations:              200039   AIC:                         2.510e+05
Df Residuals:                  200035   BIC:                         2.511e+05
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------

very small p-value

There is statistically significant evidence that treatment-effect heterogenity is associated with prior number of transactions.

## Model 3: Purchase Recency

$$
Y_i = \beta_0 + \beta_D D_i + \beta_1 \text{Recency}_i^{\mathrm{std}} + \gamma_1 (D_i \times \text{Recency}_i^{\mathrm{std}}) + \varepsilon_i
$$

In [20]:
# Standardise purchase recency
recency_mean = df["days_since_last_purchase"].mean()
recency_std = df["days_since_last_purchase"].std()

recency_z = (
    df["days_since_last_purchase"] - recency_mean
) / recency_std

# Construct regression dataset
model3_df = pd.DataFrame({
    "target": Y,
    "treatment": D,
    "recency_std": recency_z
})

model3_df["treatment_x_recency"] = model3_df["treatment"] * model3_df["recency_std"]

# Define regressors
X_lpm3 = model3_df[["treatment", "recency_std", "treatment_x_recency"]]
X_lpm3 = sm.add_constant(X_lpm3)

# Fit LPM with robust standard errors
lpm_model3 = sm.OLS(model3_df["target"], X_lpm3).fit(cov_type="HC3")

print("\n" + "=" * 50)
print("Pre-specified LPM: Pre-Treatment Purchase Recency")
print("=" * 50)
print(lpm_model3.summary())


Pre-specified LPM: Pre-Treatment Purchase Recency
                            OLS Regression Results                            
Dep. Variable:                 target   R-squared:                       0.105
Model:                            OLS   Adj. R-squared:                  0.105
Method:                 Least Squares   F-statistic:                     7755.
Date:                Sat, 24 Oct 2026   Prob (F-statistic):               0.00
Time:                        18:53:42   Log-Likelihood:            -1.2821e+05
No. Observations:              200039   AIC:                         2.564e+05
Df Residuals:                  200035   BIC:                         2.565e+05
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------

p-value = 0.002 < 0.05

There is statistically significant evidence that treatment effect heterogenity is associated with pre-treatment purchase recency.

# Summary

In [21]:
results = []

models = {
    "Total spending": (lpm_model1, "treatment_x_total_spend"),
    "Number of transactions": (lpm_model2, "treatment_x_num_transactions"),
    "Purchase recency": (lpm_model3, "treatment_x_recency")
}

for name, (model, interaction) in models.items():
    results.append({
        "Variable": name,
        "Interaction (pp per SD)": model.params[interaction] * 100,
        "Robust SE (pp)": model.bse[interaction] * 100,
        "p-value": model.pvalues[interaction]
    })

summary_df = pd.DataFrame(results)

display(summary_df.round(4))

,Variable,Interaction (pp per SD),Robust SE (pp),p-value
0,Total spending,-1.2478,0.5798,0.0314
1,Number of transactions,-1.2661,0.2423,0.0000
2,Purchase recency,0.6305,0.2063,0.0022
